# Analise de mencoes por keyword

Este notebook consulta o Neo4j e gera um grafico da quantidade de mencoes de uma keyword em um periodo.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv('.env')

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

if not NEO4J_PASSWORD:
    raise ValueError('Defina NEO4J_PASSWORD no arquivo .env antes de executar.')

In [ ]:
# Parametros de consulta
KEYWORD = 'elon musk'
DATA_INICIO = '2026-04-20'
DATA_FIM = '2026-04-30'

In [ ]:
def buscar_mencoes_por_dia(keyword, data_inicio, data_fim):
    query = '''
    MATCH (a:Article)-[:MENCIONA]->(k:Keyword)
    WHERE toLower(k.word) = toLower($keyword)
      AND date(a.publishedAt) >= date($data_inicio)
      AND date(a.publishedAt) <= date($data_fim)
    RETURN date(a.publishedAt) AS dia, count(*) AS mencoes
    ORDER BY dia
    '''

    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
        with driver.session() as session:
            result = session.run(
                query,
                keyword=keyword,
                data_inicio=data_inicio,
                data_fim=data_fim,
            )
            registros = [dict(r) for r in result]

    df = pd.DataFrame(registros)
    if df.empty:
        return pd.DataFrame(columns=['dia', 'mencoes'])

    df['dia'] = pd.to_datetime(df['dia'].astype(str))
    return df

In [ ]:
df = buscar_mencoes_por_dia(KEYWORD, DATA_INICIO, DATA_FIM)

if df.empty:
    print(f"Nenhuma mencao encontrada para '{KEYWORD}' no periodo informado.")
else:
    # Preenche datas sem mencoes com zero para manter o eixo temporal continuo
    periodo = pd.date_range(start=DATA_INICIO, end=DATA_FIM, freq='D')
    serie = (
        df.set_index('dia')
          .reindex(periodo, fill_value=0)
          .rename_axis('dia')
          .reset_index()
    )

    plt.figure(figsize=(12, 5))
    plt.plot(serie['dia'], serie['mencoes'], marker='o')
    plt.title(f"Mencoes a '{KEYWORD}' por dia")
    plt.xlabel('Data')
    plt.ylabel('Quantidade de mencoes')
    plt.grid(alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    serie